# 03 - Understand our model and the finished experiment (READ-ONLY)
This notebook **trains nothing and writes nothing**. It reads saved metadata and results, rebuilds models from checkpoints on CPU **for inference only**, and proves at the end that no checkpoint changed.

**Inputs (Kaggle → Add Input):**
| input | needed for |
|---|---|
| your code dataset (`nids/`, including `s5_explain.py`) | everything |
| `nids_results.zip` (Stage 3 run: `runs/cii/main`, scalers, manifests) | everything |
| `results_01.zip` (clean parquet splits) | *optional*: worked example on a real row, protocol counts, buffer diagnostics, paired test |

Labels used below: **[code]** = read from our source, **[run]** = read from saved outputs, **[paper]** = stated by a paper, **[inference]** = our reasoning, not a measured result.

In [ ]:
import glob, hashlib, json, os, sys
from pathlib import Path
import numpy as np, pandas as pd
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 30)

CODE_DIR = os.environ.get("NIDS_CODE") or next(
    (str(Path(p).parent) for p in glob.glob("/kaggle/input/**/s5_explain.py", recursive=True)), ".")
sys.path.insert(0, CODE_DIR)
import s5_explain as X
from common import load_json

RES = X.find_dir("runs/cii/main/config.json", "NIDS_RESULTS")      # nids_results.zip contents
PREP = X.find_dir("clean_train.parquet", "NIDS_PREPARED")           # results_01.zip contents (optional)
assert RES is not None, "attach nids_results.zip (it must contain nids_out/runs/cii/main)"
print("code:", CODE_DIR, "\nresults:", RES, "\nprepared parquet:", PREP or "not attached - row-level cells will be skipped")

def sha_all(root):  # fingerprint of every checkpoint, checked again at the end
    return {str(p.relative_to(root)): hashlib.sha256(p.read_bytes()).hexdigest() for p in sorted(Path(root).rglob("*.pt"))}
PT_BEFORE = sha_all(RES)
RUN = X.load_run(RES, "cii", "main")
ARGS = RUN["config"]["args"]
print(f"{len(PT_BEFORE)} checkpoints fingerprinted | run args:", {k: ARGS[k] for k in ("epochs0", "epochs_update", "batch", "lr", "d", "heads", "layers", "dropout", "mem_per_class", "poison_target", "poison_to", "poison_frac")})

## 1. What is our architecture called?
**An FT-Transformer-style tabular Transformer, adapted by us.** It is *not* the original TabTransformer, and *not* an exact FT-Transformer reproduction. The Python class is named `TabTransformer` (`s2_model.py`), but that name is historical and misleading.

- **Like FT-Transformer** (Gorishniy et al. 2021, §3.3): every numeric feature becomes its own token $x_jW_j+b_j$ with **per-feature** $W_j, b_j$; a learned [CLS] token is placed first; PreNorm blocks; the prediction is made from the final [CLS] vector.
- **Unlike the original TabTransformer** (Huang et al. 2020, §2): there, *only categorical* columns pass through attention; continuous features skip the Transformer and are concatenated (after LayerNorm in the reference implementation) before an MLP. There is no [CLS].
- **Our differences from FT-Transformer:** GELU FFN with $4d$ instead of ReGLU with $\tfrac43 d$; the first block keeps its first LayerNorm; the head is $\mathrm{Linear}(\mathrm{LN}(h_{CLS}))$ without FT's ReLU; the protocol token has no separate bias term; init differs; signed-log + z-score scaling instead of a quantile transform; $d=32$, 3 blocks, 4 heads.

Suggested wording for the report: *"a Transformer encoder over feature tokens in the style of FT-Transformer (Gorishniy et al., 2021)"*.

## 2. The whole pipeline at a glance
```
CICDDoS2019 CSV row (88 columns)
  │ s1_preprocess: drop 10 columns (ids, IPs, ports, timestamp, simillarhttp, inbound, label→target)
  │                drop rows with inf/NaN; canonical label from the LABEL column (never the filename)
  │                global dedupe on 78 model features (float64 hash); drop label-conflict vectors
  │                keep ALL distinct BENIGN; ≤60,000 distinct rows per attack class (bottom-k hash)
  │                split per class 60/10/10/20 → train / calib / val / test   (row id = _rid)
  ▼
s1b_experiences: E0/E1/E2 = lists of TRAIN row ids (CI or CII); scaler fit on E0 train only
                 protected rows equal to a training input after the transform → excluded (166)
  ▼
s3_continual (per batch):  x(77 numbers) ─signed-log, z-score, clip ±5→ x̃ ∈ R^77 (float32)
                           protocol ─vocabulary→ id ∈ {0,1,2,3}
  ▼
s2_model:  [CLS] + 77 numeric tokens + 1 protocol token → (B, 79, 32)
           → 3 PreNorm Transformer blocks → LayerNorm(final CLS) → Linear → logits (B, K)
           → softmax → argmax over the classes the model knows
```

## 3. Cleaning, deduplication, capping, splits, leakage checks
**[code] `s1_preprocess.py` + `common.py`**
- **Columns removed (10 of 88):** `unnamed: 0`, `flow id`, `source ip`, `destination ip`, `timestamp`, `simillarhttp` (identifiers); `source port`, `destination port` (`DROP_PORTS`); `inbound` (`DROP_INBOUND`, a direction flag that the audit showed separates benign from attacks); `label` (becomes the target). **Model inputs:** 77 numeric + `protocol` = 78.
- **Label:** `canon_label` lower-cases, strips `DrDoS_`, and maps to a canonical name. It is always read from the **label column**, because on 03-11 the filenames do not match the labels.
- **Non-finite:** a row with any ±inf/NaN feature is **dropped** (`--nonfinite drop`).
- **Global dedupe:** $h(\text{row}) = $ `hash_pandas_object` of the 78 features as float64. For each label, only the first row per hash is kept. A hash seen with **two labels** is a *conflict* and is removed from every class.
- **Cap:** BENIGN keeps every distinct row. Each attack class keeps the 60,000 distinct rows with the **smallest hash** (bottom-$k$), which gives a deterministic, uniform sample of distinct rows.
- **Split:** rows are sorted by hash and get `_rid` = position. Within each class they are permuted (seed 42) and cut 60/10/10/20.
- **Leakage checks:** no hash may occur in two splits (refused otherwise). Stage 1b then refuses or excludes rows that become equal **after the transform** (section 5).

In [ ]:
sm = load_json(RES / "split_manifest.json"); st = sm["stats"]
sched = ["BENIGN"] + [k for e in RUN["config"]["schedule"] for k in e]
tab = pd.DataFrame({"raw": st["raw_rows_by_label"], "nonfinite": st["nonfinite_rows_by_label"],
                    "distinct": st["distinct_rows_by_label"], "conflict": st["conflict_distinct_by_label"],
                    "kept": st["kept_by_label"]}).fillna(0).astype(int).loc[sched]
print("[run] Stage 1 per scheduled class:\n", tab.to_string())
print("\n[run] split sizes (before Stage 1b exclusions):\n", pd.DataFrame(sm["counts_per_split"]).loc[sched].astype(int).to_string())
print("\ncross-split identical feature vectors:", sm["cross_split_feature_overlap"], "| split fractions:", sm["split_fracs"])
feats = load_json(RES / "features.json")
print(f"\n[code/run] model features: {len(feats['numeric'])} numeric + {feats['categorical']}")
print("first 5 numeric (token order 1..5):", feats["numeric"][:5])

## 4. Experiences E0/E1/E2, CI vs CII, and what each update sees
**[code] `s1b_experiences.build_manifest`.** An experience is a **list of train row ids**, nothing else.
- With seed 42, the train BENIGN ids are permuted. **CI:** all of them go to E0. **CII:** `np.array_split` into 3 disjoint portions: 19,055 / 19,054 / 19,054.
- Attack class $k$ in experience $t$: $\min(13{,}890, n_k^{train})$ ids, sampled without replacement (`ATTACK_CAP = 13,890` = LDAP's train count). The same rng call sequence means **CI and CII use identical attack rows** (checked below).
- calib/val/test rows can never be in an experience (asserted).

**[code] `s3_continual.run_arm`.** The training set of an update attempt is $D_t \cup B$: the current experience rows plus the replay buffer, with the buffer's **stored** labels.

In [ ]:
for sc in ("ci", "cii"):
    em = load_json(RES / sc / "experience_manifest.json")
    print(f"[run] {sc}: rows per experience:", em["counts_per_experience"])
if PREP is not None:
    tr = pd.read_parquet(PREP / "clean_train.parquet", columns=["_rid", "label"]).set_index("_rid")["label"]
    zc, zi = np.load(RES / "ci" / "experience_rids.npz"), np.load(RES / "cii" / "experience_rids.npz")
    for t in range(3):
        a, b = zc[f"e{t}"], zi[f"e{t}"]
        same = set(a[tr.loc[a].to_numpy() != "BENIGN"]) == set(b[tr.loc[b].to_numpy() != "BENIGN"])
        print(f"E{t}: attack rows identical in CI and CII: {same}")
rows = []
for arm, bufs in RUN["buffers"].items():
    for t, b in bufs.items():
        g = b.groupby(["true_label", "label"]).size()
        rows.append({"arm": arm, "buffer before": f"E{t}", **{f"{k[0]}→stored {k[1]}": int(v) for k, v in g.items()}})
print("\n[run] replay buffer contents (true label → stored label):")
print(pd.DataFrame(rows).fillna(0).set_index(["arm", "buffer before"]).astype(int).to_string())
print("\n[run] rows per attempt (current + replay):",
      {f"{a} E{r['experience']} {r['attempt']}": (r["n_current"], r["n_replay"]) for a, rs in RUN["rounds"].items() for r in rs})

## 5. The exact numeric transform and the frozen scaler
**[code] `s1b_experiences.fit_scaler` / `apply_scaler`.** For numeric feature $j$ of a row:
$$\operatorname{slog}(x)=\operatorname{sign}(x)\,\ln(1+|x|),\qquad
z_j=\frac{\operatorname{slog}(x_j)-\mu_j}{\sigma_j},\qquad
\tilde x_j=\operatorname{float32}\big(\operatorname{clip}(z_j,-5,5)\big).$$
$\mu_j,\sigma_j$ are the mean and population standard deviation (numpy, ddof = 0) of $\operatorname{slog}(x_j)$ over **E0 train rows only**. For CII that is 19,055 BENIGN + 13,890 Syn = 32,945 rows. They are then **frozen** for E1 and E2. If $\sigma_j<10^{-3}$ (constant in E0), we set $\sigma_j:=1$; in CII this applies to 12 features.

These are **preprocessing statistics, not learned parameters**: they are computed once by counting, and gradient descent never touches them.

**Excluding transformed duplicates [code + run]:** two different raw rows can give the same $\tilde x$ (clip, rounding, float32). 166 protected rows whose $\tilde x$ equals a training input under the CI **or** CII scaler are never scored. Causes: clip 96, rounding inside slog 47, float32 22, standardize 1. Residual overlap is 0.

In [ ]:
sc = load_json(RES / "cii" / "scaler.json")
st_tab = X.scaler_table(sc)
print("[run] frozen scaler (first 8 of 77) - statistics, NOT learned:\n", st_tab.head(8).to_string(index=False))
print("\nconstant in E0 (std set to 1):", sc["const_in_e0"])
pe = load_json(RES / "protected_exclusions.json")
print("\n[run] excluded protected rows per split/class:",
      {sp: {k: v["excluded"] for k, v in c.items()} for sp, c in pe["by_split_class"].items()})
print("cause (first transform step, CI scaler):", {sp: d["first_step"] for sp, d in pe["diagnosis"]["ci"].items()})
if PREP is not None:
    row = X.example_row(PREP, RES, "cii", "Syn")
    print(f"\n[run] worked example: train row _rid={int(row['_rid'].iloc[0])}, label={row['label'].iloc[0]}, file={row['_src'].iloc[0]}")
    w = X.worked_example(row, sc)
    print(w[(w["clip(z, +-5)"].abs() == 5) | w["feature"].isin(["flow duration", "total fwd packets", "flow bytes/s", "fwd header length"])].to_string(index=False))

**Reading the worked example (row 80842, Syn, E0):** `bwd iat min` = 89 gives $\operatorname{slog}=\ln 90=4.4998$ and $z=(4.4998-0.7265)/0.7063=5.342$, so it is **clipped to 5.0**. `flow duration` = 113 gives $\ln 114=4.7362$ and $z=(4.7362-13.3506)/4.9073=-1.755$, which is unchanged by the clip.

Another case: `fwd header length` has impossible negative values in the raw data, down to $-2.47\times10^{11}$. For that value, $\operatorname{slog}=-26.23$ and $z=(-26.23-4.59)/1.465\approx-21$, which clips to $-5$. This is the origin of most clip collisions.

## 6. How `protocol` is encoded
**[code] `fit_scaler` builds the vocabulary on E0 train rows; `cat_codes` applies it.**
- The vocabulary sorts the distinct protocol values seen in E0 and numbers them from 1. In our run that is $\{0\to1,\ 6\to2,\ 17\to3\}$ (protocol 0, TCP, UDP). **Any value not seen in E0 maps to id 0** ("unknown").
- The id indexes a learned table $E\in\mathbb R^{4\times 32}$ (`nn.Embedding(4, 32)`), giving the token $e = E[\text{id}]$. This is a **lookup, not a multiplication by the number 6 or 17**: protocol is treated as a category.
- In the scheduled classes, every train/val/test row has protocol 0, 6 or 17, so the unknown id is **never used** in this run. Protocol is very informative: Syn is about 99.96% TCP (6), UDP about 99.9% UDP (17).

In [ ]:
if PREP is not None:
    print(X.protocol_example(row, sc))
    for sp in ("train", "val", "test"):
        d = pd.read_parquet(PREP / f"clean_{sp}.parquet", columns=["label", "protocol"])
        d = d[d["label"].isin(sched)]
        print(sp, d.groupby("label")["protocol"].value_counts().unstack(fill_value=0).to_dict("index"))
else:
    print("vocabulary [run]:", sc["cat_vocab"], "| unknown → 0 | table rows:", len(sc["cat_vocab"]["protocol"]) + 1)

## 7. One batch, and the model line by line (`s2_model.py`)
**A batch before the model [code `s3_continual.train`]:** `X[b]` float32 $(B,77)$ scaled numbers; `C[b]` int64 $(B,1)$ protocol ids; `Y[b]` int64 $(B,)$ head indices in arrival order (`BENIGN, Syn, LDAP, UDP`); per-row weight $s_i$ $(B,)$. $B=512$ (the last batch of an epoch is smaller). The order is reshuffled every epoch.

**FeatureTokenizer** (lines 31-46):
- line 36-37: `W`, `b` $\in\mathbb R^{77\times32}$ are **one row per feature, not shared**.
- line 42: `x_num.unsqueeze(-1) * self.W + self.b` broadcasts $(B,77,1)\cdot(77,32)+(77,32)\to(B,77,32)$, i.e. $t_{i,j}=\tilde x_{i,j}\,W_j+b_j$. The bias $b_j$ also acts as a **feature identity**: there is no positional encoding, so it is $W_j, b_j$ that tell the model which feature a token is.
- line 39, 43: one learned `cls` $\in\mathbb R^{1\times1\times32}$, copied to every row, **position 0**.
- line 38, 45: protocol token $E[\text{id}]$, appended **last**.
- line 46: $H^{(0)}=[\,\text{cls};\ t_{i,1..77};\ E[c_i]\,]\in\mathbb R^{B\times79\times32}$. Token order: CLS, the 77 features in `features.json` order, protocol.

**Encoder** (lines 55-58): 3 × `nn.TransformerEncoderLayer(d_model=32, nhead=4, dim_feedforward=128, dropout=0.1, activation="gelu", norm_first=True)`. For each block (PreNorm):
$$U=H+\mathrm{Drop}\big(\mathrm{MHA}(\mathrm{LN}_1(H))\big),\qquad H'=U+\mathrm{Drop}\big(W_2\,\mathrm{Drop}(\mathrm{GELU}(W_1\mathrm{LN}_2(U)+\beta_1))+\beta_2\big)$$
$$\mathrm{MHA}(Z)=[A_1V_1\,|\,\dots\,|\,A_4V_4]\,W_O+b_O,\quad A_h=\mathrm{softmax}\!\Big(\tfrac{Q_hK_h^\top}{\sqrt{8}}\Big),\ Q_h=ZW_Q^h+\dots$$
- 4 heads × 8 dims = 32. $A_h\in\mathbb R^{79\times79}$ per row, so **every token attends to every token**, CLS included. Dropout 0.1 is applied to $A_h$, after attention and inside the FFN, **only in `train()` mode**.
- $\mathrm{LN}(u)=\gamma\odot\frac{u-\mathrm{mean}(u)}{\sqrt{\mathrm{var}(u)+10^{-5}}}+\beta$.

**Pooling and head** (lines 62-68): $z_i=\mathrm{LN}(H^{(3)}_{i,0})\in\mathbb R^{32}$ is the final **[CLS] only**. Then $\ell_i=z_iW_h^\top+b_h\in\mathbb R^{K}$, with $K=2$ (E0), 3 (E1), 4 (E2). Prediction: $\hat y_i=\arg\max_k \mathrm{softmax}(\ell_i)_k$ over the classes the model knows.

**`expand_head`** (lines 71-79): a new `Linear(32, K')`. Old rows of $W_h, b_h$ are copied, so **old logits are unchanged** before training. New rows get PyTorch's default init.

**Loss and gradients [code `train`]:**
$$c_k=\sqrt{\tfrac{N}{K\,n_k}},\quad s_i=c_{y_i}w_i,\quad L=\frac{\sum_{i\in\text{batch}}s_i\,(-\log p_{i,y_i})}{\sum_{i\in\text{batch}}s_i},\quad
\frac{\partial L}{\partial \ell_{i,k}}=\frac{s_i\,(p_{i,k}-\mathbb 1[k=y_i])}{\sum_j s_j}.$$
- $n_k$ counts the **stored** labels in the whole attempt's training set, $N=\sum_k n_k$, and $w_i=1$ except R1 replay rows.
- Back-propagation reaches **every** parameter (nothing is frozen in C1/R1).
- Optimizer: AdamW (lr $10^{-3}$, weight decay $10^{-4}$ on all parameters), with a fresh optimizer per attempt.
- fp16 autocast + GradScaler on CUDA. Fixed epochs; no early stopping, no LR schedule, no validation-based model selection.

In [ ]:
m, cls = X.model_from_ckpt(RUN["dir"] / "poisoned_repair" / "A2.pt", ARGS)
print("[run] final model classes (head order):", cls)
print(X.shape_trace(m, B=ARGS["batch"]).to_string(index=False))
pt = X.parameter_table(m)
print("\n[code/run] parameters LEARNED by gradient descent:")
print(pt.groupby("role")["n_params"].sum().sort_values(ascending=False).to_string(), "\nTOTAL", pt["n_params"].sum())
print("\nNOT learned (preprocessing statistics, fixed after E0): 77 means + 77 stds, clip=5, protocol vocabulary",
      sc["cat_vocab"]["protocol"])
print("\nattention weight shapes in block 1:", {n: tuple(p.shape) for n, p in m.encoder.layers[0].self_attn.named_parameters()})

## 8. Our model vs the three papers
| | TabTransformer (Huang et al. 2020) | FT-Transformer (Gorishniy et al. 2021) | Base IDS paper (arXiv 2608.04602) | **Ours** (`s2_model.py`) |
|---|---|---|---|---|
| categorical | column embedding, part shared column-id + value part (§2) | $b_j+e_j^\top W_j$ lookup + per-feature bias (§3.3) | "embeds each categorical feature" (§3.3); repo uses the `tab_transformer_pytorch` package | `nn.Embedding` lookup, id 0 = unknown, no extra bias |
| numerical | **not tokenized**; concatenated after the Transformer (§2) | token $b_j+x_jW_j$, per-feature (§3.3) | "concatenates … with the numerical features after layer normalisation" (§3.3); repo: package LayerNorms continuous features, concatenates after the Transformer | token $x_jW_j+b_j$, per-feature |
| tokens in attention | categorical only | all features + [CLS] | categorical only (per repo/package) | [CLS] + 77 numeric + protocol = 79 |
| CLS / pooling | none; flatten contextual embeddings | [CLS] first, use $T_L^{[CLS]}$ | "pooled transformer output", method not stated | [CLS] at position 0, $\mathrm{LN}(h_{CLS})$ |
| block | post-LN, FFN 4× (§2) | PreNorm, ReGLU, first LN removed in block 1 | package default | PreNorm (PyTorch), GELU 4×, first LN kept |
| classifier | MLP (hidden 4l, 2l) | $\mathrm{Linear}(\mathrm{ReLU}(\mathrm{LN}(\cdot)))$ | MLP head (§3.3/3.5 are inconsistent) | $\mathrm{Linear}(\mathrm{LN}(\cdot))$ |
| pretraining | optional MLM/RTD, semi-supervised (§2) | none | none | none |
| size | d=32, 6 layers, 8 heads (§3) | default d=192, 3 blocks, 8 heads (App. E) | d=32, 6 blocks, 10 heads (Table 5) | d=32, 3 blocks, 4 heads |
| **match** | concept only | **closest; approximate** | different (continuous features outside attention) | — |

**Borrowed:** the feature tokenizer, [CLS] pooling and PreNorm blocks (FT-Transformer); experience replay, CI/CII scenarios, head expansion that copies old rows, and buffer label flipping (base paper).

**Our decisions:** frozen E0 scaler (the base repo refits `StandardScaler` per experience in `ci_builder.py`); signed-log + clip; global dedupe and transformed-duplicate exclusion; fixed 2,000 rows/class per accepted experience instead of $b\%$; class-weighted CE; the **acceptance gate**, the **frozen teacher T** and **R1 repair** (none of these exist in the base paper); a one-class targeted 50% flip instead of a 100% flip to random classes.

*Section numbers are from the arXiv HTML versions read on 2026-10-21. Check them against the PDF you cite.*

## 9. E0, A0, T, the buffer, poisoning, C1, the gate, R1 and the fallback
**[code] `s3_continual.py`**
1. **E0:** train on E0 (10 epochs). The **same** weights are saved twice, as `T.pt` (teacher) and `A0.pt` (first accepted model).
2. **T is frozen:** it is loaded, put in `eval()`, `requires_grad=False`, and never given to an optimizer. A SHA-256 of its state is taken before the arms and checked after, together with the file on disk.
3. **Buffer $B$:** after an experience is **accepted**, up to 2,000 rows per class of that experience are added. They are sampled uniformly without replacement (rng seed 43, the same in every arm), and the **true** label is stored. Rows stay in the buffer. A rejected update adds nothing.
4. **Poisoning (poisoned arms, before E1 only):** the stored label of $\mathrm{round}(0.5\times2000)=1000$ of the 2,000 Syn rows is set to BENIGN (rng seed 44, same rows in both poisoned arms). The flips **persist** into E2.
5. **C1:** copy the last accepted model, expand the head, train 5 epochs on $D_t\cup B$ with stored labels.
6. **Gate** on protected **val** rows. $r_k(M)$ = recall of class $k$ for model $M$, and $A$ = the last accepted model evaluated on the same rows. Accept iff **all** of:
$$r_k(C)\ \ge\ r_k(A)-0.05\ \ \forall k\in\text{old attacks};\qquad r_k(C)\ge0.80\ \ \forall k\in\text{new};$$
$$\mathrm{FAR}_B=\frac{|\{i:y_i=B,\ \hat y_i\neq B\}|}{|\{i:y_i=B\}|}\le0.05;\qquad
\mathrm{FAR}_{new}=\frac{|\{i:y_i\notin\text{new},\ \hat y_i\in\text{new}\}|}{|\{i:y_i\notin\text{new}\}|}\le0.05.$$
   BENIGN recall is covered by $\mathrm{FAR}_B=1-r_B$.
7. **R1** (`poisoned_repair` only, only if C1 is rejected): restart from the last accepted model, not from C1. Each replay row gets weight $w_i=p_T(\tilde y_i\mid x_i)$ if T knows its stored label $\tilde y_i$ (BENIGN or Syn), else 1. Current rows keep weight 1. Then the same training and the same gate.
8. **Fallback:** if no attempt passes, keep the last accepted model and leave the buffer unchanged. Those new classes are then never learned.

In [ ]:
print("[run] T sha256:", RUN["manifest"]["T_sha256"][:16], "… T_unchanged =", RUN["manifest"]["T_unchanged"], "| T classes:", RUN["manifest"]["T_classes"])
branch = {"E0 train, save T and A0": True,
          "poison applied (poisoned arms)": all(RUN["buffers"][a][1]["flipped"].sum() == 1000 for a in ("poisoned", "poisoned_repair")),
          "C1 attempts": sum(r["attempt"] == "C1" for rs in RUN["rounds"].values() for r in rs),
          "C1 rejected by the gate": sum(not r["accepted"] for rs in RUN["rounds"].values() for r in rs),
          "R1 attempts (repair)": sum(r["attempt"] == "R1" for rs in RUN["rounds"].values() for r in rs),
          "fallback (no accepted candidate)": 0 if all(any(r["accepted"] for r in rs if r["experience"] == t) for rs in RUN["rounds"].values() for t in (1, 2)) else "yes",
          "buffer grew after acceptance": all(len(RUN["buffers"][a][2]) > len(RUN["buffers"][a][1]) for a in RUN["buffers"])}
print("\n[run] which coded branches were EXERCISED in this run:")
for k, v in branch.items():
    print(f"  {k:38s} {v}")
for arm, rs in RUN["rounds"].items():
    print(f"\n[run] gate recomputed from saved metrics - {arm} (asserted equal to the saved decision):")
    print(X.gate_check(rs, ARGS).to_string(index=False))

## 10. The completed experiment (CII, BENIGN+Syn → LDAP → UDP, 10 + 5 epochs, 50% Syn→BENIGN)

In [ ]:
print(pd.read_csv(RUN["dir"] / "report" / "final_test.csv").drop(columns=["test_rows_sha256"]).set_index("arm").T.to_string())
fig = X.plot_recall(RUN)

### Why the poison changed so little: confidence, not decisions
**[inference] weighted cross-entropy with contradictory labels.** Take an input that appears $n$ times labelled Syn and $n$ times labelled BENIGN. Its share of the loss, $-c_S\,n\log p_S-c_B\,n\log p_B$, is minimised at $p_S^*=\frac{c_S}{c_S+c_B}$. In the poisoned E1 attempt the stored counts are BENIGN 22,054, LDAP 13,890 and Syn 1,000. That gives $c_S=3.509$, $c_B=0.747$, so $p_S^*=0.824>0.5$, and **argmax stays Syn**.

Flipped and clean rows are different inputs, only similar to each other, so this is an idealisation. The measurements below check it.

In [ ]:
em_cii = load_json(RES / "cii" / "experience_manifest.json")["counts_per_experience"]
cnt = X.training_label_counts(RUN, em_cii, "poisoned", 1)
print("[run] stored-label counts, poisoned E1:", cnt, "| class weights:", {k: round(v, 3) for k, v in X.class_weights(cnt).items()})
print("[inference] idealised optimum p(Syn) on contradicted inputs:", round(X.contradiction_optimum(cnt), 3))
if PREP is not None:
    probs = X.buffer_probabilities(RES, PREP, RUN)
    print("\n[run, inference-only diagnostic] E1 buffer rows:")
    print(X.poison_diagnostics(probs).to_string(index=False))
    fig2 = X.plot_confidence(probs)
    for k in ("Syn", "BENIGN"):
        print(f"\n[run] paired exact McNemar test, final models clean vs poisoned, test class {k}:", X.paired_test(RES, PREP, RUN, k))
else:
    print("attach results_01.zip to compute the buffer diagnostics and the paired tests")

**What these numbers mean** (values from our CPU inference on the saved checkpoints; it reproduces the saved test recalls 0.99539595 and 0.99497740 exactly):
- **T as a repair signal (diagnostic only; R1 never ran):** T's mean $p_T(\text{stored label})$ is ≈0.004 on the 1,000 flipped rows versus ≈0.997–0.999 on clean rows. If R1 had run, flipped rows would have been down-weighted by roughly 200×.
- **A1 poisoned** still predicts the true label (Syn) for 99.6% of the flipped rows. But its mean $p(\text{Syn})$ on true-Syn buffer rows drops from ≈0.994 (A1 clean) to ≈0.79, close to the idealised 0.82. **The poison eroded confidence, and the recall-only gate cannot see confidence.**
- **Paired test on the same 11,946 Syn test rows:** clean is right on 5 rows where poisoned is wrong, and the reverse happens 0 times (exact McNemar $p=0.0625$, not significant at 0.05). On BENIGN, poisoned is better (16 vs 2, $p\approx0.0013$). This test covers only the sampling of test rows: with **one training seed**, it says nothing about run-to-run training variance.

## 11. What we can and cannot conclude
**Can (for this exact setting: CII, seed 42, Syn→BENIGN 50% of 2,000 Syn exemplars, class-weighted CE, fresh benign each experience):**
1. The full pipeline ran on real CICDDoS2019-derived data with the leakage controls active. T stayed bit-identical. All arms were scored on the same 46,527 test rows.
2. Every C1 passed the gate, by wide margins. The poison did **not** change gate decisions, and changed final Syn test recall by 0.00042 (5 of 11,946 rows; not significant on the test sample).
3. The poison **did** change the model: confidence on true-Syn replay rows fell from ≈0.994 to ≈0.79, as the weighted-loss arithmetic predicts.
4. T clearly separates flipped from clean stored labels (0.004 vs ≈0.998), so the *signal* R1 relies on is present on this data.

**Cannot:**
- that R1 repairs anything (it never ran on real data), or that the gate detects poisoning (it never rejected);
- robustness to stronger or different attacks (higher fractions, CI, other targets, all-class random flips as in the base paper);
- anything across seeds (one run), about deployment chronology (random splits), or about other datasets;
- that "poisoned has a lower benign FAR" is a real effect (single seed);
- any comparison of these numbers with the base paper.

## 12. Why the base paper's CICIDS2017 numbers are not comparable
| aspect | base paper | ours |
|---|---|---|
| dataset | CICIDS2017, 8 classes, 1,138,612 flows | CICDDoS2019 subset: BENIGN, Syn, LDAP, UDP; 95,272 distinct benign + 13,890 per attack in training |
| features | 78 CICFlowMeter features, some treated as categorical (full list not stated) | 77 numeric + protocol; ports, IPs, timestamp and `inbound` removed |
| cleaning | inf/NaN dropped; no deduplication step found in the text I read | global dedupe, label-conflict removal, transformed-duplicate exclusion |
| split | 60/20/20 | 60/10/10/20, per class, random |
| scaling | StandardScaler fit per experience (repo) | signed-log + z-score fit on E0, frozen |
| model | `tab_transformer_pytorch` TabTransformer, 6 blocks | FT-style tokenizer, 3 blocks |
| schedule | 4 experiences × 2 classes | 3 experiences × 1 attack class |
| replay | b% of each class (1/5/10%) | 2,000 rows/class/experience |
| poisoning | 100% of buffer labels flipped to random other classes, CI (Table 10) | 50% of ONE class (Syn) flipped to BENIGN, CII |
| decision rule | none (no gate, no teacher) | recall/FAR gate, teacher-weighted R1 |
| metrics | accuracy, macro-F1, forgetting, intransigence | per-class recall, benign FAR, new-class FAR |

A different dataset, classes, attack, metric and decision rule each change the numbers. Equal-looking accuracies would still not be evidence of equal performance.

## 13. Study order (paper section → our code)
1. **FT-Transformer §3.3** (Feature Tokenizer, Transformer, prediction) → `s2_model.py` lines 31-68, then section 7 above.
2. **TabTransformer §2** (column embedding, where continuous features go) → compare with `s2_model.FeatureTokenizer.forward`.
3. **Base paper §3.3, §3.5 (Table 5)** → why our model is *not* theirs (section 8).
4. **Base paper §3.6 + Table 4 (CI/CII), §3.4 (replay)** → `s1b_experiences.build_manifest`, `s3_continual.buffer_add`.
5. **Base paper §4 (preprocessing)** → `s1_preprocess.stream_dedupe_cap`, `s1b_experiences.fit_scaler/apply_scaler`, `protected_exclusions`.
6. **Base paper §3.8 + Table 10 (label flipping)** → `s3_continual.poison`, and how our threat model differs.
7. **Our additions** → `s3_continual.gate`, `teacher_weights`, `run_arm` (sections 9-11).
8. **FT-Transformer §4.3 and Appendix E** (preprocessing, default hyperparameters) → why ours differ.

## 14. Five faculty questions
1. **"Is this TabTransformer?"** No. Numeric features are tokenized per feature and attended to together with a [CLS] token, as in FT-Transformer §3.3. The original TabTransformer puts only categorical columns through attention. The closest name is "FT-Transformer-style".
2. **"How do you rule out leakage?"**
   - Global dedupe on the 78 model features before capping, and label-conflict removal.
   - The split is made before anything is fitted, and the scaler and vocabulary are fit on E0 train only.
   - Protected rows that equal a training input after the transform are excluded (166).
   - Ports, IPs, timestamp and `inbound` are removed.
   - Every run hashes its inputs.
   - Residual risk: near-duplicates under random splits.
3. **"Did your defence work?"**
   - Not tested on real data: no C1 was rejected, so R1 never ran.
   - What we can show is that T's weights would have separated poisoned rows (0.004 vs ≈0.998).
   - The poison reduced confidence without changing decisions, so a recall-only gate cannot see it.
4. **"Why freeze the scaler instead of refitting per experience?"**
   - Replayed and new rows must share one coordinate system for the same weights $W_j, b_j$. Refitting moves old classes' inputs under a fixed model.
   - Cost: new classes outside E0's range are clipped (the clip collisions show this happens).
5. **"Why is your accuracy not comparable to the paper's ~0.999?"**
   - Different dataset, classes, features, dedupe, scaler, model, replay budget, attack, metrics, and a gate.
   - Our near-1.0 recalls also reflect that these four CICDDoS2019 classes are easy to separate after dedupe (e.g. protocol nearly separates Syn from UDP).

## 15. Open points (need code inspection or a new experiment; not guessed)
1. The base paper's numeric handling is ambiguous in its text; the repo uses `tab_transformer_pytorch` (`>=0.4.2`). The exact package version and options used for the paper's tables were not verified.
2. The base paper's CICIDS2017 categorical column list (`catfeaturelist.npy`) was not inspected.
3. **Seeds:** one run. Training variance needs ≥3 seeds per arm.
4. **R1 on real data:** needs a setting where C1 is rejected (higher `poison_frac`, CI scenario, another target) or a forced-R1 ablation. The idealised optimum for CI at 50% is $p_S^*\approx0.63$, which is still Syn, so it may not trigger either.
5. **Gate blind spot:** a confidence/calibration term (e.g. mean $p$ of old classes on val) is untested. The `calib` split is unused by the current code.
6. **Shortcut risk:** negative (invalid) `fwd header length` occurs in 7.0% LDAP / 9.8% UDP / 0.26% BENIGN / 0% Syn train rows, and protocol nearly separates Syn from UDP. No ablation has been run.
7. `fwd header length.1` is identical to `fwd header length` in 100% of train rows: a redundant token whose effect has not been tested.
8. Random flow-level splits allow near-duplicates across splits; a time-ordered split has not been run.
9. The CI scenario was prepared but never trained.
10. Epoch counts are fixed: no early stopping and no learning curve on val.

In [ ]:
PT_AFTER = sha_all(RES)
assert PT_AFTER == PT_BEFORE, "a checkpoint changed!"
print(f"read-only check: all {len(PT_AFTER)} checkpoints unchanged (sha256).")